# **Lab 02 - Introduction to Q-learning**

##### Copyright by UIT-NC@NT549

## **Some instructions before getting started**:
<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">

Start the Kernel: At the top right, choose <strong>Select Kernel ➞ Python Environments...</strong>

You can run all code blocks to check: From the menu bar, choose <strong>Run All</strong>.

Complete all code blocks marked with the comment <span style="font-family: monospace; font-weight: bold; color:white; background-color: green;"> ### YOU NEED TO WRITE YOUR CODE BELOW ### </span>
</div>

## Part 1: Implementing Q-learning with the FrozenLake-v1 environment

In [ ]:
# import Gymnasium library and alias as gym
import gymnasium as gym
import random
import numpy as np
import csv
import matplotlib.pyplot as plt
import os
import time
from collections import deque
from IPython.display import clear_output

In [ ]:
# ============================================================
# Q-learning agent for Gymnasium FrozenLake-v1
# ============================================================

class QLearningAgent:
    """Q-learning agent using a sparse dictionary Q-table."""

    def __init__(self, env, alpha=0.1, gamma=0.9):
        self.env = env
        self.alpha = alpha   # Learning rate
        self.gamma = gamma   # Discount factor
        self.q_table = {}    # {(state, action): q_value}

    # ----------------------------------------------------------
    def get_q_value(self, state, action):
        """Return Q(s, a); default 0.0 for unseen pairs."""
        return self.q_table.get((state, action), 0.0)

    # ----------------------------------------------------------
    def update_q_value(self, state, action, reward, next_state):
        """
        Q-learning update:
          Q(s,a) <- Q(s,a) + alpha * [r + gamma * max_a' Q(s',a') - Q(s,a)]
        """
        # 1) Best possible future value
        best_next_q = max(
            self.get_q_value(next_state, a)
            for a in range(self.env.action_space.n)
        )
        # 2) Current estimate
        current_q = self.get_q_value(state, action)
        # 3) TD target
        target_q = reward + self.gamma * best_next_q
        # 4) Blended update
        new_q = current_q + self.alpha * (target_q - current_q)
        # 5) Store
        self.q_table[(state, action)] = new_q

    # ----------------------------------------------------------
    def choose_action(self, state, epsilon=0.1):
        """Epsilon-greedy: explore with prob epsilon, exploit otherwise."""
        # 1) Exploration
        if random.random() < epsilon:
            return self.env.action_space.sample()
        # 2) Exploitation — pick best Q, break ties randomly
        q_values = [self.get_q_value(state, a) for a in range(self.env.action_space.n)]
        max_q    = max(q_values)
        best_actions = [a for a, q in enumerate(q_values) if q == max_q]
        return random.choice(best_actions)

    # ----------------------------------------------------------
    def choose_greedy_action(self, state):
        """Pure greedy (epsilon=0)."""
        q_values = [self.get_q_value(state, a) for a in range(self.env.action_space.n)]
        max_q    = max(q_values)
        best_actions = [a for a, q in enumerate(q_values) if q == max_q]
        return random.choice(best_actions)


# ─── Utility helpers ───────────────────────────────────────────

def export_q_table_to_txt(agent, file_path="q_table_final.txt"):
    """Export learned Q-values to a CSV-like text file."""
    with open(file_path, "w", encoding="utf-8") as f:
        f.write("State,Action,Q-value\n")
        for (state, action) in sorted(agent.q_table.keys()):
            q_val = agent.q_table[(state, action)]
            f.write(f"{state},{action},{q_val:.6f}\n")
    print(f"Q-table exported to: {file_path}")


def show_final_result_text(agent):
    """One greedy evaluation episode in ANSI text mode."""
    demo_env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="ansi")
    state, _ = demo_env.reset()
    print("=== FINAL RUN WITH GREEDY POLICY (TEXT) ===")
    print(demo_env.render())
    done, step, reward = False, 0, 0
    while not done:
        action = agent.choose_greedy_action(state)
        next_state, reward, terminated, truncated, _ = demo_env.step(action)
        done = terminated or truncated
        step += 1
        print(f"Step {step} - Action: {action}, Reward: {reward}")
        print(demo_env.render())
        state = next_state
    print("Agent reached the goal!" if reward > 0 else "Agent failed to reach the goal.")
    demo_env.close()


def show_final_result_pygame(agent, delay_seconds=0.5):
    """Greedy evaluation episode with GUI (requires local display)."""
    demo_env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="human")
    state, _ = demo_env.reset()
    print("=== FINAL RUN WITH GREEDY POLICY (PYGAME/HUMAN) ===")
    done, step, reward = False, 0, 0
    while not done:
        action = agent.choose_greedy_action(state)
        next_state, reward, terminated, truncated, _ = demo_env.step(action)
        done = terminated or truncated
        step += 1
        print(f"Step {step} - Action: {action}, Reward: {reward}")
        state = next_state
        time.sleep(delay_seconds)
    print("Agent reached the goal!" if reward > 0 else "Agent failed to reach the goal.")
    time.sleep(1.0)
    demo_env.close()


# ─── Training loop ─────────────────────────────────────────────

if __name__ == "__main__":
    # Create deterministic FrozenLake environment
    env = gym.make("FrozenLake-v1", is_slippery=False)

    # Initialize agent
    agent = QLearningAgent(env)

    num_episodes = 1000
    ql_rewards   = []   # track reward per episode (Q-learning)
    rnd_rewards  = []   # track reward per episode (random baseline)

    # ── Q-learning training ──
    for _ in range(num_episodes):
        state, _ = env.reset()
        done = False
        ep_reward = 0.0

        while not done:
            action = agent.choose_action(state, epsilon=0.1)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            agent.update_q_value(state, action, reward, next_state)
            ep_reward += reward
            state = next_state

        ql_rewards.append(ep_reward)

    # ── Random baseline (same number of episodes) ──
    for _ in range(num_episodes):
        state, _ = env.reset()
        done = False
        ep_reward = 0.0
        while not done:
            action = env.action_space.sample()
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_reward += reward
            state = next_state
        rnd_rewards.append(ep_reward)

    # ── Plot comparison ──
    window = 50
    ql_smooth  = np.convolve(ql_rewards,  np.ones(window)/window, mode='valid')
    rnd_smooth = np.convolve(rnd_rewards, np.ones(window)/window, mode='valid')

    plt.figure(figsize=(10, 4))
    plt.plot(ql_smooth,  label="Q-learning", color="steelblue")
    plt.plot(rnd_smooth, label="Random",     color="tomato", linestyle="--")
    plt.xlabel("Episode")
    plt.ylabel(f"Reward (rolling avg {window})")
    plt.title("FrozenLake — Q-learning vs Random Policy")
    plt.legend()
    plt.tight_layout()
    plt.savefig("frozen_lake_comparison.png", dpi=120)
    plt.show()

    # ── Export & demo ──
    print("Training complete. Learned Q-table:")
    for (s, a) in sorted(agent.q_table.keys()):
        print(f"  State {s}, Action {a}: {agent.q_table[(s,a)]:.2f}")

    export_q_table_to_txt(agent, "q_table_final.txt")
    show_final_result_text(agent)
    env.close()


## Part 2: Q-learning with Custom Environment "VacuumCleaner"

In [ ]:
# ============================================================
# Part 2 — VacuumCleaner Custom Environment
# ============================================================

class VacuumCleanerEnv(gym.Env):
    """
    Grid-world vacuum-cleaner environment.
    Action space  : Discrete(4) → {0:up, 1:down, 2:left, 3:right}
    Observation   : Dict('position': (row,col), 'dust': m×n binary grid)
    Rewards       : +1.0 clean dirty cell | -0.5 revisit clean cell
                    -10.0 hit obstacle (episode ends)
                    +10.0 all cells clean (episode ends)
    """

    def __init__(self, m=5, n=5, obstacle=(2, 2)):
        super(VacuumCleanerEnv, self).__init__()
        self.m = m
        self.n = n
        self.obstacle = tuple(obstacle)

        # Action space: 4 directions
        self.action_space = gym.spaces.Discrete(4)

        # Observation space
        self.observation_space = gym.spaces.Dict({
            'position': gym.spaces.Box(
                low=np.array([0, 0], dtype=np.int32),
                high=np.array([m - 1, n - 1], dtype=np.int32),
                dtype=np.int32
            ),
            'dust': gym.spaces.Box(
                low=0, high=1, shape=(m, n), dtype=np.int32
            )
        })

        self.reset()

    # ----------------------------------------------------------
    def reset(self, *, seed=None, options=None):
        """Reset robot to top-left corner; fill room with dust."""
        # Robot starts at (0, 0)
        self.position  = np.array([0, 0], dtype=np.int32)
        # Every cell is dirty (1) except the obstacle
        self.dust_grid = np.ones((self.m, self.n), dtype=np.int32)
        self.dust_grid[self.obstacle] = 0   # obstacle has no dust
        self.total_reward = 0.0
        self.truncated    = False
        self.terminated   = False
        obs = {'position': self.position.copy(), 'dust': self.dust_grid.copy()}
        return obs, {}

    # ----------------------------------------------------------
    def step(self, action):
        """Move robot, vacuum current cell, return transition tuple."""
        # Compute candidate new position
        if action == 0:    # Up
            candidate = np.array([self.position[0] - 1, self.position[1]], dtype=np.int32)
        elif action == 1:  # Down
            candidate = np.array([self.position[0] + 1, self.position[1]], dtype=np.int32)
        elif action == 2:  # Left
            candidate = np.array([self.position[0], self.position[1] - 1], dtype=np.int32)
        elif action == 3:  # Right
            candidate = np.array([self.position[0], self.position[1] + 1], dtype=np.int32)
        else:
            candidate = self.position.copy()

        # Boundary check — stay in place if out of bounds
        if (0 <= candidate[0] < self.m) and (0 <= candidate[1] < self.n):
            # Obstacle check
            if tuple(candidate) == self.obstacle:
                self.position = candidate.copy()
                reward = -10.0
                self.terminated = True
                obs = {'position': self.position.copy(), 'dust': self.dust_grid.copy()}
                self.total_reward += reward
                return obs, reward, True, False, {}
            else:
                self.position = candidate.copy()
        # else: stay in place (no movement, no reward change for wall)

        # Vacuum current cell
        pos_key = tuple(self.position)
        if self.dust_grid[pos_key] == 1:
            reward = 1.0
            self.dust_grid[pos_key] = 0
        else:
            reward = -0.5

        self.total_reward += reward

        # Check if room fully cleaned
        if np.sum(self.dust_grid) == 0:
            reward += 10.0
            self.terminated = True

        obs = {'position': self.position.copy(), 'dust': self.dust_grid.copy()}
        return obs, reward, bool(self.terminated), bool(self.truncated), {}

    # ----------------------------------------------------------
    def render(self, mode='human'):
        try:
            clear_output(wait=True)
        except Exception:
            os.system('cls' if os.name == 'nt' else 'clear')

        display = np.full((self.m, self.n), ' ', dtype='<U1')
        for i in range(self.m):
            for j in range(self.n):
                if (i, j) == self.obstacle:
                    display[i, j] = '#'
                elif self.dust_grid[i, j] == 1:
                    display[i, j] = '.'

        x, y = int(self.position[0]), int(self.position[1])
        display[x, y] = 'X' if (x, y) == self.obstacle else 'R'

        for row in display:
            print(''.join(row))
        print(f"Total reward: {self.total_reward}")
        time.sleep(0.15)

    # ----------------------------------------------------------
    def obs_to_state(self, obs):
        """Convert dict observation to a hashable state for Q-table."""
        pos  = tuple(obs['position'])
        dust = tuple(obs['dust'].flatten())
        return (pos, dust)


In [ ]:
# ─── Baseline policies for VacuumCleaner ────────────────────────

def robot_policy(option="random", env=None, step=0, direction=1, col=0):
    """
    Select an action based on the chosen policy option.

    Parameters
    ----------
    option    : 'random' | 'round_robin' | 'priority'
    env       : VacuumCleanerEnv instance
    step      : current step count (used by round_robin)
    direction : current horizontal sweep direction (+1 or -1)
    col       : current column position (used by round_robin S-shape)
    """
    if option == "random":
        return env.action_space.sample()

    elif option == "round_robin":
        # S-shape (zig-zag) sweep:
        # • Move right/left across each row
        # • At the end of a row, move down one step
        row, col_pos = int(env.position[0]), int(env.position[1])
        if direction == 1:           # moving right
            if col_pos < env.n - 1:
                return 3             # right
            else:
                return 1             # down (then flip direction)
        else:                        # moving left
            if col_pos > 0:
                return 2             # left
            else:
                return 1             # down (then flip direction)

    elif option == "priority":
        # Wall-hugging: go straight until hitting a wall/obstacle,
        # then move inward (down).
        row, col_pos = int(env.position[0]), int(env.position[1])
        # Try to keep moving right; if at boundary or blocked, go down
        if col_pos < env.n - 1 and (row, col_pos + 1) != env.obstacle:
            return 3   # right
        elif row < env.m - 1:
            return 1   # down
        else:
            return 2   # left (backtrack when fully cornered)

    raise ValueError(f"Unsupported policy option: {option}")


In [ ]:
# ─── Train Q-learning on VacuumCleaner then compare policies ──

if __name__ == "__main__":

    # ── Helper: convert obs dict → hashable state ──────────────
    def vac_obs_to_state(obs):
        return (tuple(obs['position']), tuple(obs['dust'].flatten()))

    # ── Train QLearningAgent on VacuumCleaner ──────────────────
    train_env   = VacuumCleanerEnv(m=5, n=5, obstacle=(2, 2))
    vac_agent   = QLearningAgent(train_env, alpha=0.1, gamma=0.9)
    TRAIN_EPS   = 2000
    MAX_STEPS   = 500   # truncate long episodes

    print("Training Q-learning on VacuumCleaner …")
    for ep in range(TRAIN_EPS):
        obs, _ = train_env.reset()
        state  = vac_obs_to_state(obs)
        done   = False
        steps  = 0
        while not done and steps < MAX_STEPS:
            action     = vac_agent.choose_action(state, epsilon=max(0.05, 0.5 - ep/TRAIN_EPS))
            obs, reward, terminated, truncated, _ = train_env.step(action)
            next_state = vac_obs_to_state(obs)
            # Manual update (env action space is Discrete(4))
            vac_agent.update_q_value(state, action, reward, next_state)
            state = next_state
            done  = terminated or truncated
            steps += 1
    print("Training done.")

    # ── Evaluate 3 policies over 100 episodes ──────────────────
    N_EVAL = 100

    def run_episodes_ql(n):
        """Q-learning greedy policy."""
        rewards = []
        eval_env = VacuumCleanerEnv(m=5, n=5, obstacle=(2, 2))
        for _ in range(n):
            obs, _ = eval_env.reset()
            state  = vac_obs_to_state(obs)
            done, ep_r, steps = False, 0.0, 0
            while not done and steps < MAX_STEPS:
                action = vac_agent.choose_greedy_action(state)
                obs, r, terminated, truncated, _ = eval_env.step(action)
                state  = vac_obs_to_state(obs)
                ep_r  += r
                done   = terminated or truncated
                steps += 1
            rewards.append(ep_r)
        return rewards

    def run_episodes_policy(n, option):
        """Deterministic baseline policies."""
        rewards = []
        eval_env = VacuumCleanerEnv(m=5, n=5, obstacle=(2, 2))
        for _ in range(n):
            obs, _ = eval_env.reset()
            done, ep_r, steps = False, 0.0, 0
            direction = 1
            while not done and steps < MAX_STEPS:
                row, col = int(eval_env.position[0]), int(eval_env.position[1])
                if option == "round_robin":
                    action = robot_policy("round_robin", env=eval_env, direction=direction)
                    # Flip direction after each row is completed
                    if (direction == 1 and col == eval_env.n - 1) or                        (direction == -1 and col == 0):
                        direction *= -1
                else:
                    action = robot_policy(option, env=eval_env)
                obs, r, terminated, truncated, _ = eval_env.step(action)
                ep_r  += r
                done   = terminated or truncated
                steps += 1
            rewards.append(ep_r)
        return rewards

    ql_r  = run_episodes_ql(N_EVAL)
    rr_r  = run_episodes_policy(N_EVAL, "round_robin")
    pri_r = run_episodes_policy(N_EVAL, "priority")

    # ── Plot ───────────────────────────────────────────────────
    plt.figure(figsize=(10, 4))
    plt.plot(ql_r,  label="Q-learning",     color="steelblue")
    plt.plot(rr_r,  label="Round Robin",    color="orange", linestyle="--")
    plt.plot(pri_r, label="Priority-based", color="green",  linestyle=":")
    plt.xlabel("Episode")
    plt.ylabel("Total Reward")
    plt.title("VacuumCleaner — Policy Comparison (100 episodes)")
    plt.legend()
    plt.tight_layout()
    plt.savefig("vacuum_policy_comparison.png", dpi=120)
    plt.show()

    print(f"Avg reward — Q-learning: {np.mean(ql_r):.2f} | "
          f"Round Robin: {np.mean(rr_r):.2f} | "
          f"Priority: {np.mean(pri_r):.2f}")


In [ ]:
def save_vacuum_metrics(rewards_dict, filename="vacuum_metrics.csv"):
    """Save per-episode rewards for each policy to a CSV file."""
    with open(filename, "w", newline="", encoding="utf-8") as f:
        policies = list(rewards_dict.keys())
        writer   = csv.writer(f)
        writer.writerow(["episode"] + policies)
        for i, row in enumerate(zip(*rewards_dict.values())):
            writer.writerow([i + 1] + list(row))
    print(f"Saved vacuum metrics to {filename}")


Evaluation and Analysis

## Part 3: Q-learning with Load Balancing Problem

In [ ]:
# ============================================================
# Part 3 — Load Balancing Environment
# ============================================================

class Task:
    """A task with a unique id and required processing time (in time steps)."""
    def __init__(self, task_id: int, processing_time: int):
        self.task_id         = task_id
        self.processing_time = processing_time


class Server:
    """
    Server with:
      - one currently running task
      - a finite-capacity FIFO waiting queue
    """

    def __init__(self, queue_capacity: int):
        self.queue_capacity = queue_capacity
        self.queue          = deque()
        self.current_task   = None
        self.remaining_time = 0

    # ----------------------------------------------------------
    def run_one_step(self):
        """Advance server by one time step. Returns completed task or None."""
        completed_task = None

        # Process one time unit for the running task
        if self.current_task is not None:
            self.remaining_time -= 1
            if self.remaining_time <= 0:
                completed_task      = self.current_task
                self.current_task   = None
                self.remaining_time = 0

                # Pull next task from queue immediately
                if self.queue:
                    next_task           = self.queue.popleft()
                    self.current_task   = next_task
                    self.remaining_time = next_task.processing_time

        return completed_task

    # ----------------------------------------------------------
    def add_task(self, task: Task) -> bool:
        """
        Try to add a task.
        Returns True (accepted) or False (dropped).
        """
        # Start immediately if server is idle
        if self.current_task is None:
            self.current_task   = task
            self.remaining_time = task.processing_time
            return True

        # Enqueue if there is room
        if len(self.queue) < self.queue_capacity:
            self.queue.append(task)
            return True

        # Queue full → drop
        return False

    # ----------------------------------------------------------
    def queue_length(self) -> int:
        return len(self.queue)


In [ ]:
class LoadBalancingEnv(gym.Env):
    """
    Custom Gym environment for load balancing.

    Action  : index of the server that will receive the new task.
    State   : per-server remaining_time + queue_length, plus global time.
    Rewards :
        +2.0  task completed
        +0.5  task accepted
        -5.0  task dropped
        -0.5 × queue_length  congestion penalty per step
    """

    metadata = {"render.modes": ["human"]}

    def __init__(self, n_servers: int = 3, queue_capacity: int = 2, seed: int = None):
        super().__init__()
        self.n_servers      = n_servers
        self.queue_capacity = queue_capacity
        self.rng            = random.Random(seed)

        self.servers      = [Server(queue_capacity) for _ in range(n_servers)]
        self.time         = 0
        self.total_reward = 0.0

        # Task tracking
        self.next_task_id      = 0
        self.tasks_created     = {}
        self.tasks_completed   = set()
        self.tasks_dropped     = set()
        self.tasks_accepted    = set()

        # RL spaces
        self.action_space = gym.spaces.Discrete(n_servers)

        # Observation: for each server → (remaining_time, queue_length); plus time
        # We use a flat Box for simplicity: [rem_0, q_0, rem_1, q_1, …, time]
        low  = np.zeros(n_servers * 2 + 1, dtype=np.float32)
        high = np.full(n_servers * 2 + 1, np.inf, dtype=np.float32)
        self.observation_space = gym.spaces.Box(low=low, high=high, dtype=np.float32)

    # ----------------------------------------------------------
    def _new_task(self) -> Task:
        t = Task(self.next_task_id, self.rng.randint(1, 5))
        self.tasks_created[t.task_id] = t
        self.next_task_id += 1
        return t

    # ----------------------------------------------------------
    def reset(self, *, seed=None, options=None):
        if seed is not None:
            self.rng.seed(seed)

        # Reinitialize servers, counters, reward
        self.servers      = [Server(self.queue_capacity) for _ in range(self.n_servers)]
        self.time         = 0
        self.total_reward = 0.0

        # Clear task statistics
        self.next_task_id    = 0
        self.tasks_created   = {}
        self.tasks_completed = set()
        self.tasks_dropped   = set()
        self.tasks_accepted  = set()

        return self._get_observation(), {}

    # ----------------------------------------------------------
    def step(self, action: int):
        """
        1) Advance all servers by one time step.
        2) Generate one new task.
        3) Route task to selected server.
        4) Compute reward and return transition.
        """
        reward = 0.0

        # 1) Process running tasks on each server
        for server in self.servers:
            completed = server.run_one_step()
            if completed is not None:
                self.tasks_completed.add(completed.task_id)
                reward += 2.0

        # 2) Generate new incoming task
        new_task = self._new_task()

        # 3) Route task based on action (selected server index)
        accepted = self.servers[action].add_task(new_task)
        if accepted:
            self.tasks_accepted.add(new_task.task_id)
            reward += 0.5
        else:
            self.tasks_dropped.add(new_task.task_id)
            reward -= 5.0

        # 4) Congestion penalty proportional to queue sizes
        for server in self.servers:
            reward -= 0.5 * server.queue_length()

        # Update global counters
        self.total_reward += reward
        self.time         += 1

        terminated = False
        truncated  = False
        info       = self._get_info()

        return self._get_observation(), reward, terminated, truncated, info

    # ----------------------------------------------------------
    def _get_observation(self):
        """Flat float32 array: [rem_0, q_0, rem_1, q_1, …, time]."""
        obs = []
        for server in self.servers:
            obs.append(float(server.remaining_time))
            obs.append(float(server.queue_length()))
        obs.append(float(self.time))
        return np.array(obs, dtype=np.float32)

    # ----------------------------------------------------------
    def obs_to_state(self, obs):
        """
        Convert float observation vector to a hashable state for Q-table.
        We discretise remaining_time (0-5) and queue_length (0-capacity).
        """
        state = []
        for i in range(self.n_servers):
            rem = int(min(obs[i * 2],     5))
            q   = int(min(obs[i * 2 + 1], self.queue_capacity))
            state.append((rem, q))
        return tuple(state)

    # ----------------------------------------------------------
    def _get_info(self):
        total_created = len(self.tasks_created)
        dropped       = len(self.tasks_dropped)
        completed     = len(self.tasks_completed)
        accepted      = len(self.tasks_accepted)

        drop_rate       = dropped   / total_created if total_created > 0 else 0.0
        completion_rate = completed / total_created if total_created > 0 else 0.0

        return {
            "time":             self.time,
            "total_created":    total_created,
            "dropped":          dropped,
            "completed":        completed,
            "accepted":         accepted,
            "drop_rate":        drop_rate,
            "completion_rate":  completion_rate,
            "total_reward":     self.total_reward,
        }


In [ ]:
def load_balancing_policy(option="random", env=None):
    """Simple baseline policies for load balancing."""
    if option == "random":
        return env.action_space.sample()

    elif option == "round_robin":
        # Cycle through servers in order using global time counter
        return env.time % env.n_servers

    raise ValueError(f"Unsupported policy option: {option}")


In [ ]:
# ─── Train Q-learning on LoadBalancing then compare with Round Robin ──

if __name__ == "__main__":
    N_SERVERS  = 3
    Q_CAP      = 2
    TRAIN_STEPS = 5000
    EVAL_EPS    = 100
    EP_STEPS    = 200    # steps per episode

    # ── Q-learning training ────────────────────────────────────
    lb_env   = LoadBalancingEnv(n_servers=N_SERVERS, queue_capacity=Q_CAP, seed=42)
    lb_agent = QLearningAgent(lb_env, alpha=0.1, gamma=0.9)

    print("Training Q-learning on LoadBalancing …")
    obs, _ = lb_env.reset()
    state  = lb_env.obs_to_state(obs)
    for t in range(TRAIN_STEPS):
        eps    = max(0.05, 1.0 - t / TRAIN_STEPS)
        action = lb_agent.choose_action(state, epsilon=eps)
        obs, reward, _, _, _ = lb_env.step(action)
        next_state = lb_env.obs_to_state(obs)
        lb_agent.update_q_value(state, action, reward, next_state)
        state = next_state
    print("Training done.")

    # ── Evaluation helper ──────────────────────────────────────
    def eval_lb_policy(policy_fn, n_episodes, ep_steps):
        """Run policy_fn for n_episodes × ep_steps; return metrics list."""
        metrics_rows = []
        env = LoadBalancingEnv(n_servers=N_SERVERS, queue_capacity=Q_CAP, seed=0)
        for ep in range(n_episodes):
            obs, _ = env.reset()
            ep_reward, ep_drop, ep_complete = 0.0, 0, 0
            for s in range(ep_steps):
                action = policy_fn(obs, env)
                obs, reward, _, _, info = env.step(action)
                ep_reward   += reward
                ep_drop      = info["dropped"]
                ep_complete  = info["completed"]
            total_created = info["total_created"] if info["total_created"] > 0 else 1
            metrics_rows.append({
                "episode":          ep + 1,
                "total_reward":     ep_reward,
                "drop_rate":        ep_drop / total_created,
                "completion_rate":  ep_complete / total_created,
            })
        return metrics_rows

    def ql_policy(obs, env):
        state = env.obs_to_state(obs)
        return lb_agent.choose_greedy_action(state)

    def rr_policy(obs, env):
        return load_balancing_policy("round_robin", env=env)

    ql_metrics  = eval_lb_policy(ql_policy, EVAL_EPS, EP_STEPS)
    rr_metrics  = eval_lb_policy(rr_policy, EVAL_EPS, EP_STEPS)

    # ── Save metrics ───────────────────────────────────────────
    def save_lb_metrics(metrics, filename):
        with open(filename, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=metrics[0].keys())
            writer.writeheader()
            writer.writerows(metrics)
        print(f"Saved: {filename}")

    save_lb_metrics(ql_metrics, "lb_qlearning_metrics.csv")
    save_lb_metrics(rr_metrics, "lb_roundrobin_metrics.csv")

    # ── Comparison table ───────────────────────────────────────
    def avg(rows, key): return np.mean([r[key] for r in rows])

    print("\n=== Comparison Table ===")
    print(f"{'Metric':<20} {'Q-learning':>12} {'Round Robin':>12}")
    print("-" * 46)
    print(f"{'Average Reward':<20} {avg(ql_metrics,'total_reward'):>12.2f} {avg(rr_metrics,'total_reward'):>12.2f}")
    print(f"{'Drop Rate':<20} {avg(ql_metrics,'drop_rate'):>12.4f} {avg(rr_metrics,'drop_rate'):>12.4f}")
    print(f"{'Completion Rate':<20} {avg(ql_metrics,'completion_rate'):>12.4f} {avg(rr_metrics,'completion_rate'):>12.4f}")

    # ── Plot ───────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics_to_plot = [
        ("total_reward",    "Reward"),
        ("drop_rate",       "Drop Rate"),
        ("completion_rate", "Completion Rate"),
    ]
    for ax, (key, title) in zip(axes, metrics_to_plot):
        ql_vals  = [r[key] for r in ql_metrics]
        rr_vals  = [r[key] for r in rr_metrics]
        ax.plot(ql_vals,  label="Q-learning",  color="steelblue")
        ax.plot(rr_vals,  label="Round Robin", color="orange", linestyle="--")
        ax.set_title(title)
        ax.set_xlabel("Episode")
        ax.legend()
    plt.suptitle("LoadBalancing — Q-learning vs Round Robin (100 episodes)")
    plt.tight_layout()
    plt.savefig("lb_policy_comparison.png", dpi=120)
    plt.show()


In [ ]:
def save_metrics(metrics_list, filename="metrics.csv"):
    """
    Save a list of metric dicts to a CSV file.

    Parameters
    ----------
    metrics_list : list of dict  (all dicts must share the same keys)
    filename     : output CSV path
    """
    if not metrics_list:
        print("No metrics to save.")
        return
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=metrics_list[0].keys())
        writer.writeheader()
        writer.writerows(metrics_list)
    print(f"Metrics saved to {filename}")


Evaluation and Analysis

## CONGRATULATIONS TEAM!

Congratulations to the team for completing Lab02 - Introduction to Q-learning.
Keep up the effort in the next sections.

References: https://gymnasium.farama.org/ 

## ADDITIONAL INFORMATION

**Author**: M.Sc. Phan Trung Phat - Department of Computer Networks and Communications, UIT

**Contact**: phatpt@uit.edu.vn
